# 09 – Gold dim_customer (SCD Type 2)

Bu notebook `workspace.gold.dim_customer` tablosunu kurar: önce bronze snapshot'ından ilk yükleme, sonra landing batch'leri 1 → 3 SCD2 MERGE ile.

**Bu hücre:** Tabloyu `CREATE OR REPLACE` ile yeniden oluşturur ve bronze `client` tablosundan yükler. `batch_log`'daki `dim_customer` satırlarını siler, tekrar çalıştırılabilir.

**Çıktı:** `rows`, `current_rows` ve `clients` 5.369 olmalı; NULL sayıları 0. Client 435 için `district_id` 20 ve `is_current` true olmalı.

In [0]:
import os
import sys
import importlib

from pyspark.sql import functions as F

repo_root = os.path.dirname(os.getcwd())
src_path = os.path.join(repo_root, "src")
if src_path not in sys.path:
    sys.path.append(src_path)

import silver_transforms
import silver_merge
import gold_dim_customer
importlib.reload(silver_transforms)
importlib.reload(silver_merge)
importlib.reload(gold_dim_customer)

gold_dim_customer.load_initial(spark)

df_dim = spark.table(gold_dim_customer.GOLD_TABLE)

df_summary = df_dim.agg(
    F.count("*").alias("rows"),
    F.count_if(F.col("is_current")).alias("current_rows"),
    F.countDistinct("client_id").alias("clients"),
    F.count_if(F.col("birth_date").isNull()).alias("null_birth_date"),
    F.count_if(F.col("district_name").isNull()).alias("null_district_name"),
    F.count_if(F.col("gender") == "F").alias("female"),
    F.min("customer_sk").alias("min_sk"),
    F.max("customer_sk").alias("max_sk"),
)
display(df_summary)

display(df_dim.filter(F.col("client_id") == 435))

## SCD2 MERGE: batch 1 → 3

**Bu hücre:** Landing `client` batch'lerini sırayla SCD2 MERGE ile uygular. Değişen müşterinin açık kaydı kapanır, yeni versiyon eklenir. `batch_log`'da SUCCESS olan batch atlanır.

**Çıktı:** Her batch için SUCCESS. Log'da `rows_inserted` / `rows_updated` değerleri batch 1: 13 / 8, batch 2: 24 / 19, batch 3: 9 / 5 olmalı. Özette `rows` 5.415, `current_rows` 5.383 (silver client ile aynı), `closed_rows` 32 olmalı.

In [0]:
for batch_id in [1, 2, 3]:
    result = gold_dim_customer.run_batch(spark, batch_id)
    print(f"batch {batch_id}: {result}")

df_log = spark.table("workspace.silver.batch_log").filter(F.col("table_name") == "dim_customer")
display(df_log.orderBy("batch_id"))

df_dim = spark.table(gold_dim_customer.GOLD_TABLE)
df_summary = df_dim.agg(
    F.count("*").alias("rows"),
    F.count_if(F.col("is_current")).alias("current_rows"),
    F.count_if(~F.col("is_current")).alias("closed_rows"),
)
display(df_summary)

## Testler

**Bu hücre:** Client 435'in tarihçesini gösterir. Point-in-time sorgu yapar (`valid_from <= ts AND ts < valid_to`). Her müşterinin tek current kaydı olduğunu ve dim'in current halinin silver `client` ile aynı olduğunu kontrol eder. Son olarak batch 2'yi iki şekilde tekrar çalıştırır:

- `run_batch` ile: log batch'i atlamalı.

- Log olmadan doğrudan MERGE ile: `row_hash` ve `change_ts` koşulları değişikliği engellemeli (defense in depth).

**Çıktı:** Client 435 için 2 satır (20 → 28, 1999-02-18). `df_checks`'te bütün satırlarda `ok` true olmalı.

In [0]:
GOLD_TABLE = gold_dim_customer.GOLD_TABLE
df_dim = spark.table(GOLD_TABLE)

# 1) client 435 tarihçesi
df_435 = df_dim.filter(F.col("client_id") == 435).orderBy("valid_from")
display(df_435.select("customer_sk", "client_id", "district_id", "district_name",
                      "valid_from", "valid_to", "is_current"))


def district_at(client_id, ts_text):
    # point-in-time: valid_to exclusive
    sql_text = f"""
        SELECT district_id FROM {GOLD_TABLE}
        WHERE client_id = {client_id}
          AND valid_from <= TIMESTAMP '{ts_text}'
          AND TIMESTAMP '{ts_text}' < valid_to
    """
    rows = spark.sql(sql_text).collect()
    return rows[0]["district_id"]


rows_before = df_dim.count()
checks = []
checks.append(("435 version count", 2, df_435.count()))
checks.append(("435 district at 1999-02-10", 20, district_at(435, "1999-02-10")))
checks.append(("435 district at change_ts boundary", 28, district_at(435, "1999-02-18 02:08:09")))

# 2) her müşteride tek current, tarih aralığı geçerli
df_multi_current = df_dim.filter(F.col("is_current")).groupBy("client_id").count().filter(F.col("count") > 1)
checks.append(("clients with >1 current", 0, df_multi_current.count()))
checks.append(("rows valid_from >= valid_to", 0, df_dim.filter(F.col("valid_from") >= F.col("valid_to")).count()))

# 3) dim current = silver client (district_id)
df_silver = spark.table("workspace.silver.client").select("client_id", F.col("district_id").alias("silver_district_id"))
df_current = df_dim.filter(F.col("is_current")).select("client_id", "district_id")
df_compare = df_silver.join(df_current, on="client_id", how="left")
df_mismatch = df_compare.filter(F.col("district_id").isNull() | (F.col("district_id") != F.col("silver_district_id")))
checks.append(("silver vs dim current mismatch", 0, df_mismatch.count()))

# 4) batch 2 tekrarı, log ile: atlanmalı
rerun_status = gold_dim_customer.run_batch(spark, 2)
checks.append(("rerun batch 2 skipped (1 = yes)", 1, int(rerun_status == "SKIPPED")))

# 5) batch 2 tekrarı, log olmadan doğrudan MERGE: satır koşulları korumalı
gold_dim_customer.prepare_staged_source(spark, 2)
merge_result = spark.sql(gold_dim_customer.SCD2_MERGE_SQL).collect()[0]
changed_by_direct_merge = merge_result["num_inserted_rows"] + merge_result["num_updated_rows"]
checks.append(("direct MERGE batch 2 ins + upd", 0, changed_by_direct_merge))
checks.append(("row count unchanged", rows_before, spark.table(GOLD_TABLE).count()))

df_checks = spark.createDataFrame(checks, ["check", "expected", "actual"])
df_checks = df_checks.withColumn("ok", F.col("expected") == F.col("actual"))
display(df_checks)